# Avaliação Prática 1 — CIFAR-10 · executor Kaggle (2 GPUs)

Rota de fuga do Colab, que esgotou a cota gratuita de GPU. A do Kaggle é **independente**
(30 h/semana) — e traz uma vantagem que o Colab não tem: **`T4 x2`, duas GPUs**.

As execuções deste experimento são independentes entre si (cada uma grava um JSON com
rótulo distinto e nenhuma lê a saída da outra), então [`run_parallel.py`](run_parallel.py)
as distribui em duas pistas via `CUDA_VISIBLE_DEVICES`. Isso corta o tempo total de
**~4h20 para ~2h30**.

**Antes de rodar**, painel direito (`⋮` → Settings):

1. `Accelerator` → **GPU T4 x2** — o `x2` importa; com uma GPU só, o paralelismo não existe
2. `Internet` → **On** — sem isso o `git clone` e os pesos da ImageNet falham

> ⚠️ **Baixe o zip (última célula) a cada etapa concluída, não só no fim.** Uma sessão
> reciclada leva `/kaggle/working` junto — foi assim que perdemos 2 h de GPU no Colab.

In [ ]:
# 1. Clonar o repositório, instalar dependências, e checar o que falha em silêncio.
import pathlib, subprocess

REPO = pathlib.Path('/kaggle/working/machine-learning-fundamentals')
ACTIVITY = REPO / 'activities' / 'avaliacao-pratica-1'

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/fsd-dantas/machine-learning-fundamentals.git',
                    str(REPO)], check=True)

assert ACTIVITY.is_dir(), f'{ACTIVITY} não existe no repositório publicado.'
%cd {ACTIVITY}
!pip install -q -r requirements.txt

import tensorflow as tf, torch, transformers
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow  ', tf.__version__, f'| GPUs: {len(gpus)}')
print('PyTorch     ', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('transformers', transformers.__version__)

# Ambas as falhas abaixo são silenciosas e caras: sem GPU o experimento levaria dias;
# com transformers 5.x o ViT treina com o encoder ALEATÓRIO e reporta um número mentiroso.
assert torch.cuda.is_available(), 'Sem GPU: Settings → Accelerator → GPU T4 x2'
assert transformers.__version__ < '5', "rode: !pip install -q 'transformers>=4.40,<5'"
if len(gpus) < 2:
    print('\n⚠️  só 1 GPU visível — o paralelismo cai por terra (Settings → GPU T4 x2)')

In [ ]:
# 2. RESTAURAR resultados anteriores, se você tiver um zip de outra sessão.
#    Anexe-o em + Add Input → Upload. Sem zip, esta célula não faz nada e segue o jogo.
import pathlib, shutil, zipfile

RESULTS = pathlib.Path('results'); RESULTS.mkdir(exist_ok=True)
n = 0
for entrada in pathlib.Path('/kaggle/input').rglob('*'):
    if entrada.suffix == '.json' and entrada.name[:3] in ('s1_','s2_','s3_','s4_','s5_'):
        shutil.copy(entrada, RESULTS / entrada.name); n += 1
    elif entrada.suffix == '.zip':
        with zipfile.ZipFile(entrada) as z:
            for nome in z.namelist():
                if nome.endswith('.json') and 'results/' in nome:
                    (RESULTS / pathlib.Path(nome).name).write_bytes(z.read(nome)); n += 1

print(f'{n} artefatos recuperados · {len(list(RESULTS.glob("*.json")))} JSONs em results/')

In [ ]:
# 3. O QUE FALTA. run_all.py NÃO pula o que existe — reexecutar uma etapa retreina tudo.
!python status.py

In [ ]:
# 4. CORE — as 5 estratégias, duas pistas (~30 min em vez de 37).
#    Pareamento por custo: a s4 (18 min) roda sozinha numa GPU enquanto a outra
#    despacha as baratas. O ViT fecha sozinho, pois é o único em PyTorch.
!python run_parallel.py "s4_augmentation.py --policy flip_crop" "s1_cnn_scratch.py"
!python run_parallel.py "s3_finetuning.py --backbone mobilenetv2 --head gap" "s2_feature_extraction.py --backbone mobilenetv2 --classifier svm"
!python s5_vit.py
!python report.py

In [ ]:
# 5. ABLAÇÕES BARATAS — questões 2(a) e 4(a), uma em cada GPU (~40 min).
!python run_parallel.py "s2_feature_extraction.py --ablation" "s3_finetuning.py --ablation-head --seeds 42 7"
!python report.py

In [ ]:
# ⬇️ BAIXE AGORA. Não espere o fim. (roda em segundos)
%run -i baixar.py

In [ ]:
# 6. ABLAÇÕES CARAS — questões 4(c) e 4(b), uma em cada GPU (~80 min em vez de 160).
#    Épocas reduzidas de 15+12 para 8+6, IDÊNTICAS em todos os braços: a validade interna
#    se mantém (a pergunta é qual braço vence qual), mas as acurácias absolutas das
#    ablações não são comparáveis à tabela principal. Declarado no relatório.
!python run_parallel.py \
  "s4_augmentation.py --ablation-policy --seeds 42 7 --epochs-frozen 8 --epochs-finetune 6" \
  "s4_augmentation.py --ablation-optimizer --seeds 42 7 --epochs-frozen 8 --epochs-finetune 6"
!python report.py

In [ ]:
# 7. OPCIONAL — dispersão da tabela principal + o setup do notebook da aula (~40 min).
!python run_parallel.py "s1_cnn_scratch.py --seed 7" "s1_cnn_scratch.py --seed 2024"
!python run_parallel.py "s5_vit.py --seed 7" "s5_vit.py --seed 2024"
!python run_parallel.py "s3_finetuning.py --backbone mobilenetv2 --head gap --seed 2024" "s3_finetuning.py --backbone vgg16 --head flatten --frozen-only"
!python report.py

In [ ]:
# 8. EMPACOTAR — os JSONs são a evidência; sem eles, nenhuma GPU gasta conta.
%run -i baixar.py